# Amazon India Sales Analysis

This notebook explores the source workbook used by the Amazon India dashboard. It summarizes overall performance, monthly trends, category and product results, order outcomes, payment and fulfillment channels, and state performance.

Run the cells from top to bottom. The notebook looks for the Excel workbook beside itself, then in the parent `outputs` folder.


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

plt.style.use("seaborn-v0_8-whitegrid")
DATA_PATH = Path("Amazon Sales Data India.xlsx")
if not DATA_PATH.exists():
    DATA_PATH = Path("outputs") / DATA_PATH.name

raw = pd.read_excel(DATA_PATH)
df = raw.copy()
df["Order_Date"] = pd.to_datetime(df["Order_Date"], errors="coerce")
for column in ["Quantity", "Unit_Price_INR", "Discount_Pct", "Total_Sales_INR", "Profit_INR"]:
    df[column] = pd.to_numeric(df[column], errors="coerce")
df = df.dropna(subset=["Order_Date"]).copy()
print(f"Loaded {len(df):,} order rows from {DATA_PATH}")
display(df.head())


## Data quality and coverage

In [ ]:
print("Date range:", df["Order_Date"].min().date(), "to", df["Order_Date"].max().date())
print("Unique orders:", f"{df['Order_ID'].nunique():,}")
display(df.isna().sum().rename("Missing values").to_frame())
display(df.describe(include="all").T)


## Overall performance

In [ ]:
sales = df["Total_Sales_INR"].sum()
profit = df["Profit_INR"].sum()
orders = df["Order_ID"].nunique()
units = df["Quantity"].sum()
summary = pd.DataFrame({"Metric": ["Total sales", "Total profit", "Total orders", "Units sold", "Average order value", "Profit margin"],
    "Value": [sales, profit, orders, units, sales / orders if orders else 0, profit / sales if sales else 0]})
display(summary)
print(f"Sales: ₹{sales:,.2f} | Profit: ₹{profit:,.2f} | Margin: {profit/sales:.1%}" if sales else "No sales in this dataset.")


## Monthly sales and profit trend

In [ ]:
monthly = (df.assign(Month=df["Order_Date"].dt.to_period("M").dt.to_timestamp())
    .groupby("Month", as_index=False)[["Total_Sales_INR", "Profit_INR"]].sum())
ax = monthly.set_index("Month").plot(figsize=(12, 5), marker="o", linewidth=2)
ax.set_title("Monthly Sales and Profit")
ax.set_ylabel("INR")
ax.set_xlabel("Order month")
ax.legend(["Sales", "Profit"])
plt.tight_layout()
plt.show()


## Category and top product performance

In [ ]:
category = (df.groupby("Category", as_index=False)
    .agg(Sales=("Total_Sales_INR", "sum"), Profit=("Profit_INR", "sum"), Units_Sold=("Quantity", "sum"), Orders=("Order_ID", "nunique"))
    .sort_values("Sales", ascending=False))
display(category.style.format({"Sales": "₹{:,.0f}", "Profit": "₹{:,.0f}", "Units_Sold": "{:,.0f}", "Orders": "{:,.0f}"}))
category.set_index("Category")[["Sales", "Profit"]].plot(kind="barh", figsize=(10, 5), color=["#2D6CDF", "#16A6A1"])
plt.title("Sales and Profit by Category")
plt.xlabel("INR")
plt.tight_layout()
plt.show()

top_products = (df.groupby("Product", as_index=False)
    .agg(Sales=("Total_Sales_INR", "sum"), Profit=("Profit_INR", "sum"), Units_Sold=("Quantity", "sum"))
    .nlargest(10, "Sales"))
display(top_products.style.format({"Sales": "₹{:,.0f}", "Profit": "₹{:,.0f}", "Units_Sold": "{:,.0f}"}))


## Order status, returns, and cancellations

In [ ]:
status = (df.groupby("Order_Status", as_index=False)
    .agg(Orders=("Order_ID", "nunique"), Sales_Value=("Total_Sales_INR", "sum")))
status["Order_Share"] = status["Orders"] / orders if orders else 0
display(status.style.format({"Orders": "{:,.0f}", "Sales_Value": "₹{:,.0f}", "Order_Share": "{:.1%}"}))
status_key = df["Order_Status"].astype(str).str.casefold()
returned = df.loc[status_key.eq("returned"), "Order_ID"].nunique()
cancelled = df.loc[status_key.eq("cancelled"), "Order_ID"].nunique()
print(f"Return rate: {returned/orders:.1%} | Cancellation rate: {cancelled/orders:.1%}" if orders else "No orders in this dataset.")
print("Returned/cancelled sales values are indicators of recorded order value, not confirmed accounting losses.")

by_category_status = (df.assign(Status_Key=status_key)
    .groupby(["Category", "Status_Key"])["Order_ID"].nunique().unstack(fill_value=0))
for column in ["returned", "cancelled"]:
    if column not in by_category_status: by_category_status[column] = 0
by_category_status["Return_Share"] = by_category_status["returned"] / by_category_status.sum(axis=1).replace(0, pd.NA)
by_category_status["Cancellation_Share"] = by_category_status["cancelled"] / by_category_status.sum(axis=1).replace(0, pd.NA)
display(by_category_status[["returned", "cancelled", "Return_Share", "Cancellation_Share"]].sort_values("Return_Share", ascending=False).style.format({"Return_Share": "{:.1%}", "Cancellation_Share": "{:.1%}"}))


## Payment, fulfillment, and state performance

In [ ]:
def performance_table(group):
    return (df.groupby(group, as_index=False)
        .agg(Sales=("Total_Sales_INR", "sum"), Profit=("Profit_INR", "sum"), Orders=("Order_ID", "nunique"))
        .sort_values("Sales", ascending=False))

for dimension in ["Payment_Method", "Fulfillment"]:
    print(dimension.replace("_", " "))
    display(performance_table(dimension).style.format({"Sales": "₹{:,.0f}", "Profit": "₹{:,.0f}", "Orders": "{:,.0f}"}))

states = performance_table("Ship_State")
display(states.head(15).style.format({"Sales": "₹{:,.0f}", "Profit": "₹{:,.0f}", "Orders": "{:,.0f}"}))
states.head(15).set_index("Ship_State")[["Sales", "Profit"]].plot(kind="barh", figsize=(10, 6), color=["#2D6CDF", "#16A6A1"])
plt.title("Top 15 States by Sales: Sales and Profit")
plt.xlabel("INR")
plt.tight_layout()
plt.show()
